# MIRA-Bench annotation

Downloads the official `ziqiangoodgood/MIRAGE` dataset repository and annotates the 100 MIRA-Bench samples. Its JSONL contains each composite instruction and five per-edit polygon masks. For each mask, the LLM parses the full instruction with that region highlighted and selects the corresponding source reference; masks ground the regions but do not replace LLM parsing. Each parsed reference becomes a visual token (`<v1>`–`<v5>`). Images have no edited target image, so `target_ref` remains null. The MIRA-Bench paper and source repository are available at [arXiv](https://arxiv.org/abs/2604.05180) and [GitHub](https://github.com/ZiqianLiu666/MIRAGE).

Install from the repository root with `uv sync`, copy `.env.example` to `.env`, and set `RANDOM_SEED` plus `GEMINI_API_KEY` (or `GOOGLE_API_KEY`). New runs use `.out/annotation/mira_bench/run_<timestamp>/`. Existing canonical runs are resumed automatically; if only legacy `.out/annotation/mirage/` runs exist, the latest is resumed in place using its existing filenames and stable sample IDs.

In [1]:
import json
import os
from dataclasses import replace
from functools import partial
from pathlib import Path

from huggingface_hub import snapshot_download

from annotation import (
    AnnotationConfig,
    convert_rows,
    create_annotation_run_directory,
    mira_bench,
    resolve_annotation_run_directory,
)

DATASET_ID = "ziqiangoodgood/MIRAGE"
SPLIT = "test"
CONFIG = AnnotationConfig.from_env("mira_bench")
LEGACY_OUTPUT_DIR = CONFIG.output_dir.parent / "mirage"
LEGACY_RUNS = sorted(
    path
    for path in LEGACY_OUTPUT_DIR.glob("run_*")
    if path.is_dir() and (path / "mirage_mixed.jsonl").is_file()
)
USE_LEGACY_OUTPUT = not os.environ.get("ANNOTATE_OUT") and not any(
    path.is_dir() for path in CONFIG.output_dir.glob("run_*")
) and bool(LEGACY_RUNS)
if USE_LEGACY_OUTPUT:
    CONFIG = replace(CONFIG, output_dir=LEGACY_OUTPUT_DIR)
OUTPUT_NAME = "mirage_mixed.jsonl" if USE_LEGACY_OUTPUT else "mira_bench_mixed.jsonl"
AUDIT_NAME = "mirage_audit.jsonl" if USE_LEGACY_OUTPUT else "mira_bench_audit.jsonl"

DATASET_SNAPSHOT = Path(
    snapshot_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        allow_patterns=[
            "benchmark/annotations.jsonl",
            "benchmark/*.jpg",
            "benchmark/*.jpeg",
            "benchmark/*.png",
            "benchmark/*.webp",
        ],
    )
)
BENCHMARK_ROOT = DATASET_SNAPSHOT / "benchmark"
ANNOTATION_FILE = BENCHMARK_ROOT / "annotations.jsonl"
with ANNOTATION_FILE.open(encoding="utf-8") as stream:
    ROWS = [json.loads(line) for line in stream if line.strip()]

if not ROWS:
    raise ValueError(f"No MIRA-Bench annotations found in {ANNOTATION_FILE}")
print(f"Loaded {len(ROWS)} MIRA-Bench rows from {ANNOTATION_FILE}")

/Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 601 files: 100%|██████████| 601/601 [00:00<00:00, 7658.30it/s]

Loaded 100 MIRA-Bench rows from /Users/nghialt3670/.cache/huggingface/hub/datasets--ziqiangoodgood--MIRAGE/snapshots/11eff1e3f396e189e61bd1f0ca596286d8a0b183/benchmark/annotations.jsonl


The released MIRA-Bench JSONL stores `image`, a composite `editing_instruction`, and five ordered target-region polygons in `mask`; it does not include `bbox` or `refer_object` fields. The adapter builds a separate mask per edit. The LLM parses the full instruction with each mask highlighted; if a parse is empty, ambiguous, or does not align to the instruction, it retries once with MIRA-Bench-specific guidance to select the object containing a masked part. Each region mask is retained for its reference crop and mask asset. Attempts are capped by `ANNOTATE_LIMIT` from `.env` (defaults to 10 per run; set it to `0` to process all remaining rows).

In [2]:
RESUME_LAST_RUN = any(
    path.is_dir() for path in CONFIG.output_dir.glob("run_*")
)
RUN_DIR = (
    resolve_annotation_run_directory(CONFIG.output_dir)
    if RESUME_LAST_RUN
    else create_annotation_run_directory(CONFIG.output_dir)
)
RUN_CONFIG = replace(CONFIG, output_dir=RUN_DIR)
print("Annotation run directory:", RUN_DIR)

summary = convert_rows(
    rows=ROWS,
    adapter=partial(mira_bench, image_root=BENCHMARK_ROOT),
    split=SPLIT,
    annotator=RUN_CONFIG.create_annotator(use_grounder=False),
    output=RUN_DIR / OUTPUT_NAME,
    failures=RUN_DIR / AUDIT_NAME,
    limit=RUN_CONFIG.limit,
)
summary

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Annotation run directory: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/annotation/mirage/run_20261015_210158_946635


test: 100it [15:33,  9.34s/it]


{'accepted': 13, 'skipped': 0, 'failed': 0, 'resumed': 87, 'retried': 13}